[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/46_sft_data.ipynb)

#  Medium: SFT Data Construction

Build a **Supervised Fine-Tuning dataset**: format instruction-response pairs, tokenize, and create labels with instruction masking for loss computation.

### Signature
```python
def build_sft_dataset(conversations, tokenizer, max_length=2048):
    """Build SFT dataset from conversation data.
    Args:
        conversations: List[{"instruction": str, "response": str}]
        tokenizer: HuggingFace-style tokenizer with __call__() returning {"input_ids", "attention_mask"}
        max_length: max sequence length (truncate if longer)
    Returns:
        dict with keys "input_ids" (B, L) and "labels" (B, L)
            labels: instruction tokens = -100, response tokens = original token IDs, padding = -100
    """
    ...
```

### Rules
- Concatenate instruction + response, tokenize together
- Create labels where instruction tokens are -100 (ignored in loss)
- Response tokens keep their original IDs
- Pad/truncate to `max_length`
- Return batched tensors

### Example
```python
>>> convs = [{"instruction": "What is 2+2?", "response": "It is 4."}]
>>> result = build_sft_dataset(convs, tokenizer, max_length=128)
>>> result["input_ids"].shape
torch.Size([1, 128])
>>> result["labels"][0, :3]  # instruction tokens
tensor([-100, -100, -100])
```

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
#  YOUR IMPLEMENTATION HERE

def build_sft_dataset(conversations, tokenizer, max_length=2048):
    """Build SFT dataset from conversation data."""
    pass  # Replace this


In [ ]:
#  Test your implementation

class SimpleTokenizer:
    def __init__(self):
        self.pad_token_id = 0
        self.eos_token_id = 2
    def __call__(self, text, return_tensors=None, padding=False, truncation=False, max_length=None):
        ids = [ord(c) for c in text] + [self.eos_token_id]
        if max_length and len(ids) > max_length:
            ids = ids[:max_length-1] + [self.eos_token_id]
        return {"input_ids": torch.tensor(ids), "attention_mask": [1]*len(ids)}

tok = SimpleTokenizer()
convs = [{"instruction": "Hello", "response": "World"}]
result = build_sft_dataset(convs, tok, max_length=32)

print("input_ids shape:", result["input_ids"].shape)
print("labels shape:", result["labels"].shape)

# Check instruction tokens are masked
labels = result["labels"][0]
print("First 5 labels:", labels[:5])
assert "input_ids" in result and "labels" in result
print(" Passed!")

In [ ]:
#  SUBMIT  Run this cell to check your solution
from torch_judge import check
check("sft_data")